# scPoli 三级注释流程 Python版

基于二校正结果 `cell_type_level2` 做 subset/recluster，并用 marker 得到三级注释列 `cell_type_level3`。
直接用原来的scpoli来

该文件的输出都在：/home/lixiangyu/zr/Annotate/ANNOTATE_new/9_annotate_level3/0514_rename_level3/output_mouse/work_0514

# import

In [1]:
import anndata
import os
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
sc.settings.verbosity=2
sc.settings.seed=1234
np.random.seed(1234)

In [2]:
outdir="/home/lixiangyu/zr/Annotate/ANNOTATE_new/8_annotate_level2/0514_rename_level2/output_mouse/work_0514_ori"
adata = sc.read_h5ad(os.path.join(outdir,"scPoli_concat_level2_marker_allmouse.h5ad"))
adata

AnnData object with n_obs × n_vars = 561576 × 57692
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'tissue', 'cell_type_level1', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'atlas_key', 'cell_type_level1_corrected', 'cell_type_level2'
    var: 'original_gene_names'
    obsm: 'X_scPoli', 'X_umap', 'decontX_GSE131776_10_UMAP', 'decontX_GSE131776_11_UMAP', 'decontX_GSE131776_12_UMAP', 'decontX_GSE131776_13_UMAP', 'decontX_GSE131776_14_UMAP', 'decontX_GSE131776_15_UMAP', 'decontX_GSE131776_16_UMAP', 'decontX_GSE131776_17_UMAP', 'decontX_GSE131776_18_UMAP', 'decontX_GSE131776_1_UMAP', 'decontX_GSE131776_2_UMAP', 'decontX_GSE131776_3_U

In [3]:
adata.obs['cell_type_level2'].value_counts()

cell_type_level2
Fibroblast                               115916
Smooth muscle cell                       111592
Homeostatic/Resident macrophage           71132
CD4 T cell                                67882
B cell                                    26250
Arterial endothelial cell                 25138
Matrix-remodeling/SMC-like macrophage     21074
Inflammatory macrophage                   18777
Neutrophil                                17705
Fibromyocyte                              16691
Other macrophage                          14838
Classical monocyte                        11161
CD8 T cell                                 8486
Erythrocyte/Erythroid                      7748
cDC1                                       7036
Natural killer cell                        5678
Intermediate monocyte                      5193
Foamy macrophage                           2560
Plasma cell                                1839
Pericyte                                   1293
Lymphatic endothelial c

In [4]:
adata.obsm["X_scPoli"].shape

(561576, 10)

In [ ]:
# sc.pp.neighbors(adata, n_neighbors=15, use_rep="X_scPoli")
# sc.tl.umap(adata)

computing neighbors
    finished (0:03:26)
computing UMAP
    finished (0:23:11)


In [ ]:
# ##合并后先画一张全体细胞的 UMAP，检查 cell_type_level1_corrected 是否正常
# outdir="../0514_rename_level2/output_allhuman/work_0514/"
# if "X_umap" in adata.obsm and "cell_type_level1_corrected" in adata.obs:
#     sc.pl.umap(adata,color="cell_type_level1_corrected",legend_loc="on data",frameon=False,size=1,show=False)
#     plt.savefig(os.path.join(outdir,"cell_type_level1_corrected.pdf"),bbox_inches="tight")
#     plt.close()
# adata

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


AnnData object with n_obs × n_vars = 1015699 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'neighbors', 'umap', 'cell_type_level1_corrected_colors'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
    obsp: 'distances', 'connectivities'

# EC-Arterial

In [5]:
label_col = "cell_type_level2"
cluster_key = "leiden_scpoli_res3"
outdir="./output_mouse/work_0514/ec_arterial/"

In [6]:
def subset_normalize_recluster_scpoli(adata, target_cell_types, outdir, prefix=None, label_col="cell_type_level2", counts_layer="counts", lognorm_layer="lognorm", scpoli_rep="X_scPoli", cluster_key=None, n_neighbors=150, leiden_resolution=3, leiden_random_state=1234, umap_min_dist=1.1, umap_random_state=1234, metric="euclidean", target_sum=1e4, size=2, save_h5ad=True):
    os.makedirs(outdir, exist_ok=True)
    target_cell_types = list(target_cell_types)
    if prefix is None:
        prefix = "_".join([x.lower().replace(" ", "_").replace("/", "_").replace("+", "pos") for x in target_cell_types])
    if cluster_key is None:
        cluster_key = f"leiden_scpoli_res{leiden_resolution}"
    if label_col not in adata.obs:
        raise ValueError(f"{label_col} not found in adata.obs")
    if counts_layer not in adata.layers:
        raise ValueError(f"{counts_layer} not found in adata.layers")
    if scpoli_rep not in adata.obsm:
        raise ValueError(f"{scpoli_rep} not found in adata.obsm")

    print("Target cell types:", target_cell_types)
    print("Original object:", adata)
    work = adata[adata.obs[label_col].astype(str).isin(target_cell_types), :].copy()
    print("Subset object:", work)
    print(f"{label_col}.value_counts() after subsetting:")
    print(work.obs[label_col].value_counts())
    if work.n_obs == 0:
        raise ValueError("No cells selected. Please check target_cell_types and label_col.")

    work.X = work.layers[counts_layer].copy()
    work.layers[counts_layer] = work.X.copy()
    sc.pp.normalize_total(work, target_sum=target_sum)
    sc.pp.log1p(work)
    work.layers[lognorm_layer] = work.X.copy()

    x = work.layers[lognorm_layer]
    v = x.data if sp.issparse(x) else np.ravel(x)
    v = v[:min(len(v), 1000000)]
    print("lognorm min", np.min(v), "max", np.max(v), "integer-like should be False", np.allclose(v, np.round(v)))
    print(f"{label_col}.value_counts() after normalization:")
    print(work.obs[label_col].value_counts())

    print("Formal correction/reclustering:")
    print(f"Only using {label_col}, {scpoli_rep}, and marker-based level2 annotation; not using cell_type_ori / cell_type_level1 as ground truth.")
    print("work object:", work)
    print(f"{label_col}.value_counts() in work:")
    print(work.obs[label_col].value_counts())

    sc.pp.neighbors(work, use_rep=scpoli_rep, n_neighbors=n_neighbors, metric=metric)
    sc.tl.leiden(work, resolution=leiden_resolution, key_added=cluster_key, random_state=leiden_random_state)
    sc.tl.umap(work, min_dist=umap_min_dist, random_state=umap_random_state)

    for c in [cluster_key, label_col]:
        sc.pl.umap(work, color=c, legend_loc="on data", frameon=False, size=size, show=False)
        fig_path = os.path.join(outdir, f"umap_{prefix}_{c}.pdf")
        plt.savefig(fig_path, bbox_inches="tight")
        plt.close()
        print(f"Saved UMAP: {fig_path}")

    if "dataset" in work.obs:
        sc.pl.umap(work, color="dataset", frameon=False, size=size, show=False)
        fig_path = os.path.join(outdir, f"umap_{prefix}_dataset.pdf")
        plt.savefig(fig_path, bbox_inches="tight")
        plt.close()
        print(f"Saved UMAP: {fig_path}")

    print(f"{cluster_key}.value_counts().sort_index():")
    print(work.obs[cluster_key].value_counts().sort_index())

    if save_h5ad:
        save_path = os.path.join(outdir, f"{prefix}_scPoli_recluster_umap.h5ad")
        work.write_h5ad(save_path)
        print(f"Saved to: {save_path}")

    return work

In [7]:
ec_arterial = subset_normalize_recluster_scpoli(
    adata=adata,
    target_cell_types=["Arterial endothelial cell"],
    outdir=outdir,
    prefix="ec_arterial_level3",
    cluster_key="leiden_scpoli_res3"
)

Target cell types: ['Arterial endothelial cell']
Original object: AnnData object with n_obs × n_vars = 561576 × 57692
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'tissue', 'cell_type_level1', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'atlas_key', 'cell_type_level1_corrected', 'cell_type_level2'
    var: 'original_gene_names'
    obsm: 'X_scPoli', 'X_umap', 'decontX_GSE131776_10_UMAP', 'decontX_GSE131776_11_UMAP', 'decontX_GSE131776_12_UMAP', 'decontX_GSE131776_13_UMAP', 'decontX_GSE131776_14_UMAP', 'decontX_GSE131776_15_UMAP', 'decontX_GSE131776_16_UMAP', 'decontX_GSE131776_17_UMAP', 'decontX_GSE131776_18_UMAP', 'decontX_GSE

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/preprocessing/_normalization.py:196: UserWarning: Some cells have zero counts
  warn(UserWarning('Some cells have zero counts'))


    finished (0:00:05)
lognorm min 0.35060126 max 8.470931 integer-like should be False False
cell_type_level2.value_counts() after normalization:
cell_type_level2
Arterial endothelial cell    25138
Name: count, dtype: int64
Formal correction/reclustering:
Only using cell_type_level2, X_scPoli, and marker-based level2 annotation; not using cell_type_ori / cell_type_level1 as ground truth.
work object: AnnData object with n_obs × n_vars = 25138 × 57692
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'tissue', 'cell_type_level1', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'atlas_key', 'cell_type_level1_corrected', 'cell_type_level2

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_mouse/work_0514/ec_arterial/umap_ec_arterial_level3_leiden_scpoli_res3.pdf


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_mouse/work_0514/ec_arterial/umap_ec_arterial_level3_cell_type_level2.pdf


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_mouse/work_0514/ec_arterial/umap_ec_arterial_level3_dataset.pdf
leiden_scpoli_res3.value_counts().sort_index():
leiden_scpoli_res3
0     1182
1     1057
2     1000
3      999
4      907
5      907
6      884
7      822
8      801
9      792
10     790
11     774
12     751
13     735
14     726
15     725
16     715
17     699
18     694
19     683
20     675
21     667
22     662
23     658
24     657
25     653
26     635
27     601
28     588
29     508
30     502
31     446
32     416
33     401
34     390
35      36
Name: count, dtype: int64
Saved to: ./output_mouse/work_0514/ec_arterial/ec_arterial_level3_scPoli_recluster_umap.h5ad


In [ ]:
outdir="./output_mouse/work_0514/ec_arterial/"
work = sc.read_h5ad(os.path.join(outdir, "ec_arterial_level3_scPoli_recluster_umap.h5ad"))
work

AnnData object with n_obs × n_vars = 21668 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected', 'cell_type_level2', 'leiden_scpoli_res3'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'cell_type_level2_colors', 'dataset_colors', 'leiden', 'leiden_scpoli_res3_colors', 'log1p', 'neighbors', 'umap'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'lognorm', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
    obsp: 'connectivities', 'distances'

In [28]:
# 如果 h5ad 里 var_names 不是 gene symbol，先改成 gene symbol
work.var_names = work.var["original_gene_names"].astype(str).values
work.var_names_make_unique()

# marker-based sanity check：用 marker 给每个细胞打 marker_top 标签
marker_dict = {
   "Quiescent state endothelial": ['KLF2', 'KLF4', 'NOS3', 'THBD', 'ENG'],##, 'RBPJ' 'PTPRB',
    "Pro-angiogenic endothelial, tip state": ['ESM1', 'APLN', 'ANGPT2', 'KDR', 'DLL4', 'CXCR4', 'PGF', 'SOX17'], 
    "Pro-angiogenic endothelial, stalk state":['JAG1', 'HES1', 'PTPRB', 'NOTCH1', 'RBPJ', 'HEY1', 'HEY2'],
    "EndoMT":['COL1A1', 'COL1A2', 'TAGLN', 'ACTA2', 'FN1', 'SNAI1', 'SNAI2'],
    "Inflammatory endothelial": ['SELE', 'ICAM1', 'VCAM1', 'NFKBIA', 'RELA', 'IL6', 'CCL2', 'CXCL8']
}

marker_dict = {k: [g for g in v if g in work.var_names] for k, v in marker_dict.items()}
marker_dict = {k: v for k, v in marker_dict.items() if len(v) > 0}

print("marker genes used:")
for k, v in marker_dict.items():
    print(k, v)

def mean_expr_by_genes(adata, genes, layer="lognorm"):
    X = adata[:, genes].layers[layer] if layer in adata.layers else adata[:, genes].X
    return np.asarray(X.mean(axis=1)).ravel()

score_cols = []
score_col_to_label = {}

for ct, genes in marker_dict.items():
    safe = ct.replace(" ", "_").replace("/", "_")
    col = f"marker_score__{safe}"
    work.obs[col] = mean_expr_by_genes(work, genes, layer="lognorm")
    score_cols.append(col)
    score_col_to_label[col] = ct

if len(score_cols) > 0:
    score_mat = work.obs[score_cols].to_numpy(dtype=float)
    order = np.argsort(score_mat, axis=1)
    top_idx = order[:, -1]
    second_idx = order[:, -2] if len(score_cols) > 1 else order[:, -1]

    score_labels = np.array([score_col_to_label[c] for c in score_cols])
    work.obs["marker_top"] = score_labels[top_idx]
    work.obs["marker_top_score"] = score_mat[np.arange(work.n_obs), top_idx]
    work.obs["marker_second_score"] = score_mat[np.arange(work.n_obs), second_idx]
    work.obs["marker_margin"] = work.obs["marker_top_score"] - work.obs["marker_second_score"]
    work.obs["marker_disagree"] = ((work.obs["marker_top"].astype(str) != work.obs[label_col].astype(str)) & (work.obs["marker_margin"].astype(float) > 0.1))

    sc.pl.dotplot(work, var_names=marker_dict, groupby=label_col, standard_scale="var", show=False)
    plt.savefig(os.path.join(outdir, "dot_marker_ec_arterial_by_level3_marker.pdf"), bbox_inches="tight")
    plt.close()

    sc.pl.dotplot(work, var_names=marker_dict, groupby=cluster_key, standard_scale="var", show=False)
    plt.savefig(os.path.join(outdir, "dot_marker_ec_arterial_by_cluster.pdf"), bbox_inches="tight")
    plt.close()
else:
    work.obs["marker_top"] = "Unknown"
    work.obs["marker_margin"] = np.nan
    work.obs["marker_disagree"] = False
    print("no marker genes found; skip marker sanity check")

marker genes used:
Quiescent state endothelial ['KLF2', 'KLF4', 'NOS3', 'THBD', 'ENG']
Pro-angiogenic endothelial, tip state ['ESM1', 'APLN', 'ANGPT2', 'KDR', 'DLL4', 'CXCR4', 'PGF', 'SOX17']
Pro-angiogenic endothelial, stalk state ['JAG1', 'HES1', 'PTPRB', 'NOTCH1', 'RBPJ', 'HEY1', 'HEY2']
EndoMT ['COL1A1', 'COL1A2', 'TAGLN', 'ACTA2', 'FN1', 'SNAI1', 'SNAI2']
Inflammatory endothelial ['SELE', 'ICAM1', 'VCAM1', 'NFKBIA', 'RELA', 'IL6', 'CCL2', 'CXCL8']


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  obs_bool.groupby(level=0).sum() / obs_bool.groupby(level=0).count()
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:178: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dot_color_df = self.obs_tidy.groupby(level=0).mean()
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:747: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap', 'norm' will be ignored
  dot

In [29]:
# cluster-level 二级注释：不用预测标签多数派，只看每个 scPoli cluster 内的 marker_top 多数派
if "marker_top" in work.obs and (work.obs["marker_top"].astype(str) != "Unknown").any():
    marker_tab = pd.crosstab(work.obs[cluster_key].astype(str), work.obs["marker_top"].astype(str))
    marker_frac = marker_tab.div(marker_tab.sum(axis=1), axis=0)
    cluster_major_marker = marker_tab.idxmax(axis=1)
    cluster_major_marker_frac = marker_frac.max(axis=1)
    n_cells = marker_tab.sum(axis=1)
else:
    clusters = work.obs[cluster_key].astype(str).value_counts().sort_index().index
    cluster_major_marker = pd.Series("Unknown", index=clusters)
    cluster_major_marker_frac = pd.Series(np.nan, index=clusters)
    n_cells = work.obs[cluster_key].astype(str).value_counts().reindex(clusters).fillna(0).astype(int)

cluster_summary = pd.DataFrame({
    "cluster": cluster_major_marker.index.astype(str),
    "major_marker": cluster_major_marker.values,
    "major_marker_frac": cluster_major_marker_frac.reindex(cluster_major_marker.index).values,
    "n_cells": n_cells.reindex(cluster_major_marker.index).values,
}).set_index("cluster")

# marker-only 结果：不再使用预测标签多数派
cluster_summary["cluster_label_clean"] = cluster_summary["major_marker"].astype(str)
cluster_summary.loc[cluster_summary["cluster_label_clean"].eq("Unknown"), "cluster_label_clean"] = "Uncertain"

cluster_summary = cluster_summary.copy()
cluster_summary["cluster_num"] = cluster_summary.index.astype(int)
cluster_summary = cluster_summary.sort_values("cluster_num")

print(cluster_summary.drop(columns="cluster_num").to_string())
cluster_summary.drop(columns="cluster_num").to_csv(os.path.join(outdir, "ec_arterial_cluster_level3_marker_summary.csv"))


                        major_marker  major_marker_frac  n_cells          cluster_label_clean
cluster                                                                                      
0        Quiescent state endothelial           0.743860     1140  Quiescent state endothelial
1        Quiescent state endothelial           0.699458     1108  Quiescent state endothelial
2        Quiescent state endothelial           0.750249     1005  Quiescent state endothelial
3        Quiescent state endothelial           0.722166      997  Quiescent state endothelial
4        Quiescent state endothelial           0.780543      884  Quiescent state endothelial
5        Quiescent state endothelial           0.765714      875  Quiescent state endothelial
6        Quiescent state endothelial           0.789655      870  Quiescent state endothelial
7        Quiescent state endothelial           0.855491      865  Quiescent state endothelial
8        Quiescent state endothelial           0.761501     

In [31]:
corrected_annotation = {
    "2": "Inflammatory endothelial",
    "9": "Inflammatory endothelial",
    "12": "Inflammatory endothelial",
    "25": "Inflammatory endothelial"
}
# marker-only 二级注释结果：不再写回/覆盖一级列 cell_type_level1_corrected
level2_col = "cell_type_level3"
cluster_clean_map = cluster_summary["cluster_label_clean"].to_dict()
work.obs[level2_col] = work.obs[cluster_key].astype(str).map(cluster_clean_map).fillna("Unknown")

# 如需人工覆盖 marker-only 结果，再取消下面几行注释。
manual_map = work.obs[cluster_key].astype(str).map(corrected_annotation)
manual_mask = manual_map.notna()
work.obs.loc[manual_mask, level2_col] = manual_map.loc[manual_mask]

print(work.obs[level2_col].value_counts())

cell_type_level3
Quiescent state endothelial    18492
Inflammatory endothelial        3079
EndoMT                            97
Name: count, dtype: int64


In [32]:
work.write_h5ad(os.path.join(outdir, "ec_arterial_level3_scPoli_recluster_umap.h5ad"))

In [34]:
# 注释后 UMAP：看最终 corrected label
sc.pl.umap(
    work,
    color="cell_type_level3",
    legend_loc="on data",
    frameon=False,
    size=2,
    show=False
)
plt.savefig(os.path.join(outdir, "umap_ec_arterial_level3_cell_type_level3_marker.pdf"), bbox_inches="tight")
plt.close()

# # 注释后 UMAP：和 cluster 对照
# sc.pl.umap(
#     work,
#     color=cluster_key,
#     legend_loc="on data",
#     frameon=False,
#     size=2,
#     show=False
# )
# plt.savefig(os.path.join(outdir, "umap_mac_level2_cluster_after_correction.pdf"), bbox_inches="tight")
# plt.close()

# 注释后 UMAP：如果有 dataset，也看 batch/dataset 分布
if "dataset" in work.obs:
    sc.pl.umap(
        work,
        color="dataset",
        frameon=False,
        size=2,
        show=False
    )
    plt.savefig(os.path.join(outdir, "umap_ec_arterial_level3_dataset_after_correction.pdf"), bbox_inches="tight")
    plt.close()

# 注释后 dotplot：按最终 corrected label 分组
sc.pl.dotplot(
    work,
    var_names=marker_dict,
    groupby="cell_type_level3",
    standard_scale="var",
    show=False
)
plt.savefig(os.path.join(outdir, "dot_marker_ec_arterial_level3_by_level3_marker.pdf"), bbox_inches="tight")
plt.close()

# # 注释后 dotplot：按 cluster 分组，方便检查每个 cluster 的 marker
# sc.pl.dotplot(
#     work,
#     var_names=marker_dict,
#     groupby=cluster_key,
#     standard_scale="var",
#     show=False
# )
# plt.savefig(os.path.join(outdir, "dot_marker_mac_level2_by_cluster_after_correction.pdf"), bbox_inches="tight")
# plt.close()

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categoric

# Mac-infla

In [8]:
label_col = "cell_type_level2"
cluster_key = "leiden_scpoli_res3"
outdir="./output_mouse/work_0514/mac_infla/"

In [9]:
def subset_normalize_recluster_scpoli(adata, target_cell_types, outdir, prefix=None, label_col="cell_type_level2", counts_layer="counts", lognorm_layer="lognorm", scpoli_rep="X_scPoli", cluster_key=None, n_neighbors=150, leiden_resolution=3, leiden_random_state=1234, umap_min_dist=1.1, umap_random_state=1234, metric="euclidean", target_sum=1e4, size=2, save_h5ad=True):
    os.makedirs(outdir, exist_ok=True)
    target_cell_types = list(target_cell_types)
    if prefix is None:
        prefix = "_".join([x.lower().replace(" ", "_").replace("/", "_").replace("+", "pos") for x in target_cell_types])
    if cluster_key is None:
        cluster_key = f"leiden_scpoli_res{leiden_resolution}"
    if label_col not in adata.obs:
        raise ValueError(f"{label_col} not found in adata.obs")
    if counts_layer not in adata.layers:
        raise ValueError(f"{counts_layer} not found in adata.layers")
    if scpoli_rep not in adata.obsm:
        raise ValueError(f"{scpoli_rep} not found in adata.obsm")

    print("Target cell types:", target_cell_types)
    print("Original object:", adata)
    work = adata[adata.obs[label_col].astype(str).isin(target_cell_types), :].copy()
    print("Subset object:", work)
    print(f"{label_col}.value_counts() after subsetting:")
    print(work.obs[label_col].value_counts())
    if work.n_obs == 0:
        raise ValueError("No cells selected. Please check target_cell_types and label_col.")

    work.X = work.layers[counts_layer].copy()
    work.layers[counts_layer] = work.X.copy()
    sc.pp.normalize_total(work, target_sum=target_sum)
    sc.pp.log1p(work)
    work.layers[lognorm_layer] = work.X.copy()

    x = work.layers[lognorm_layer]
    v = x.data if sp.issparse(x) else np.ravel(x)
    v = v[:min(len(v), 1000000)]
    print("lognorm min", np.min(v), "max", np.max(v), "integer-like should be False", np.allclose(v, np.round(v)))
    print(f"{label_col}.value_counts() after normalization:")
    print(work.obs[label_col].value_counts())

    print("Formal correction/reclustering:")
    print(f"Only using {label_col}, {scpoli_rep}, and marker-based level2 annotation; not using cell_type_ori / cell_type_level1 as ground truth.")
    print("work object:", work)
    print(f"{label_col}.value_counts() in work:")
    print(work.obs[label_col].value_counts())

    sc.pp.neighbors(work, use_rep=scpoli_rep, n_neighbors=n_neighbors, metric=metric)
    sc.tl.leiden(work, resolution=leiden_resolution, key_added=cluster_key, random_state=leiden_random_state)
    sc.tl.umap(work, min_dist=umap_min_dist, random_state=umap_random_state)

    for c in [cluster_key, label_col]:
        sc.pl.umap(work, color=c, legend_loc="on data", frameon=False, size=size, show=False)
        fig_path = os.path.join(outdir, f"umap_{prefix}_{c}.pdf")
        plt.savefig(fig_path, bbox_inches="tight")
        plt.close()
        print(f"Saved UMAP: {fig_path}")

    if "dataset" in work.obs:
        sc.pl.umap(work, color="dataset", frameon=False, size=size, show=False)
        fig_path = os.path.join(outdir, f"umap_{prefix}_dataset.pdf")
        plt.savefig(fig_path, bbox_inches="tight")
        plt.close()
        print(f"Saved UMAP: {fig_path}")

    print(f"{cluster_key}.value_counts().sort_index():")
    print(work.obs[cluster_key].value_counts().sort_index())

    if save_h5ad:
        save_path = os.path.join(outdir, f"{prefix}_scPoli_recluster_umap.h5ad")
        work.write_h5ad(save_path)
        print(f"Saved to: {save_path}")

    return work

In [10]:
mac_infla = subset_normalize_recluster_scpoli(
    adata=adata,
    target_cell_types=["Inflammatory macrophage"],
    outdir=outdir,
    prefix="mac_infla",
    cluster_key="leiden_scpoli_res3"
)

Target cell types: ['Inflammatory macrophage']
Original object: AnnData object with n_obs × n_vars = 561576 × 57692
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'tissue', 'cell_type_level1', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'atlas_key', 'cell_type_level1_corrected', 'cell_type_level2'
    var: 'original_gene_names'
    obsm: 'X_scPoli', 'X_umap', 'decontX_GSE131776_10_UMAP', 'decontX_GSE131776_11_UMAP', 'decontX_GSE131776_12_UMAP', 'decontX_GSE131776_13_UMAP', 'decontX_GSE131776_14_UMAP', 'decontX_GSE131776_15_UMAP', 'decontX_GSE131776_16_UMAP', 'decontX_GSE131776_17_UMAP', 'decontX_GSE131776_18_UMAP', 'decontX_GSE13

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/preprocessing/_normalization.py:196: UserWarning: Some cells have zero counts
  warn(UserWarning('Some cells have zero counts'))


    finished (0:00:04)
lognorm min 0.21410197 max 8.699681 integer-like should be False False
cell_type_level2.value_counts() after normalization:
cell_type_level2
Inflammatory macrophage    18777
Name: count, dtype: int64
Formal correction/reclustering:
Only using cell_type_level2, X_scPoli, and marker-based level2 annotation; not using cell_type_ori / cell_type_level1 as ground truth.
work object: AnnData object with n_obs × n_vars = 18777 × 57692
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'tissue', 'cell_type_level1', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'atlas_key', 'cell_type_level1_corrected', 'cell_type_level2'


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_mouse/work_0514/mac_infla/umap_mac_infla_leiden_scpoli_res3.pdf


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_mouse/work_0514/mac_infla/umap_mac_infla_cell_type_level2.pdf


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_mouse/work_0514/mac_infla/umap_mac_infla_dataset.pdf
leiden_scpoli_res3.value_counts().sort_index():
leiden_scpoli_res3
0     1172
1     1167
2      906
3      898
4      873
5      861
6      841
7      810
8      806
9      709
10     697
11     695
12     681
13     671
14     634
15     615
16     590
17     550
18     542
19     470
20     460
21     448
22     440
23     402
24     381
25     377
26     372
27     367
28     342
Name: count, dtype: int64
Saved to: ./output_mouse/work_0514/mac_infla/mac_infla_scPoli_recluster_umap.h5ad


In [ ]:
label_col = "cell_type_level2"
cluster_key = "leiden_scpoli_res3"
outdir="./output_mouse/work_0514/mac_infla/"
work = sc.read_h5ad(os.path.join(outdir, "mac_infla_scPoli_recluster_umap.h5ad"))
work

AnnData object with n_obs × n_vars = 20256 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected', 'cell_type_level2', 'leiden_scpoli_res3'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'cell_type_level2_colors', 'dataset_colors', 'leiden', 'leiden_scpoli_res3_colors', 'log1p', 'neighbors', 'umap'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'lognorm', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
    obsp: 'connectivities', 'distances'

In [37]:
# 如果 h5ad 里 var_names 不是 gene symbol，先改成 gene symbol
work.var_names = work.var["original_gene_names"].astype(str).values
work.var_names_make_unique()

# marker-based sanity check：用 marker 给每个细胞打 marker_top 标签
marker_dict = {
    "Pro-inflammatory macrophage": ['SPP1', 'CD9', 'MMP12', 'TREM2', 'TNF', 'IL1B', 'NFKBIA', 'NFKBIZ', 'ITGAX'],
    "Chemokine-producing macrophage": ['CXCL2', 'CXCL8', 'CCL2', 'CCL4', 'CCL3', 'CD83', 'IRF1','IL8'],
}

marker_dict = {k: [g for g in v if g in work.var_names] for k, v in marker_dict.items()}
marker_dict = {k: v for k, v in marker_dict.items() if len(v) > 0}

print("marker genes used:")
for k, v in marker_dict.items():
    print(k, v)

def mean_expr_by_genes(adata, genes, layer="lognorm"):
    X = adata[:, genes].layers[layer] if layer in adata.layers else adata[:, genes].X
    return np.asarray(X.mean(axis=1)).ravel()

score_cols = []
score_col_to_label = {}

for ct, genes in marker_dict.items():
    safe = ct.replace(" ", "_").replace("/", "_")
    col = f"marker_score__{safe}"
    work.obs[col] = mean_expr_by_genes(work, genes, layer="lognorm")
    score_cols.append(col)
    score_col_to_label[col] = ct

if len(score_cols) > 0:
    score_mat = work.obs[score_cols].to_numpy(dtype=float)
    order = np.argsort(score_mat, axis=1)
    top_idx = order[:, -1]
    second_idx = order[:, -2] if len(score_cols) > 1 else order[:, -1]

    score_labels = np.array([score_col_to_label[c] for c in score_cols])
    work.obs["marker_top"] = score_labels[top_idx]
    work.obs["marker_top_score"] = score_mat[np.arange(work.n_obs), top_idx]
    work.obs["marker_second_score"] = score_mat[np.arange(work.n_obs), second_idx]
    work.obs["marker_margin"] = work.obs["marker_top_score"] - work.obs["marker_second_score"]
    work.obs["marker_disagree"] = ((work.obs["marker_top"].astype(str) != work.obs[label_col].astype(str)) & (work.obs["marker_margin"].astype(float) > 0.1))

    sc.pl.dotplot(work, var_names=marker_dict, groupby=label_col, standard_scale="var", show=False)
    plt.savefig(os.path.join(outdir, "dot_marker_mac_infla_by_level2.pdf"), bbox_inches="tight")
    plt.close()

    sc.pl.dotplot(work, var_names=marker_dict, groupby=cluster_key, standard_scale="var", show=False)
    plt.savefig(os.path.join(outdir, "dot_marker_mac_infla_by_cluster.pdf"), bbox_inches="tight")
    plt.close()
else:
    work.obs["marker_top"] = "Unknown"
    work.obs["marker_margin"] = np.nan
    work.obs["marker_disagree"] = False
    print("no marker genes found; skip marker sanity check")

marker genes used:
Pro-inflammatory macrophage ['SPP1', 'CD9', 'MMP12', 'TREM2', 'TNF', 'IL1B', 'NFKBIA', 'NFKBIZ', 'ITGAX']
Chemokine-producing macrophage ['CXCL2', 'CXCL8', 'CCL2', 'CCL4', 'CCL3', 'CD83', 'IRF1']


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  obs_bool.groupby(level=0).sum() / obs_bool.groupby(level=0).count()
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:178: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dot_color_df = self.obs_tidy.groupby(level=0).mean()
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:747: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap', 'norm' will be ignored
  dot

In [38]:
# cluster-level 二级注释：不用预测标签多数派，只看每个 scPoli cluster 内的 marker_top 多数派
if "marker_top" in work.obs and (work.obs["marker_top"].astype(str) != "Unknown").any():
    marker_tab = pd.crosstab(work.obs[cluster_key].astype(str), work.obs["marker_top"].astype(str))
    marker_frac = marker_tab.div(marker_tab.sum(axis=1), axis=0)
    cluster_major_marker = marker_tab.idxmax(axis=1)
    cluster_major_marker_frac = marker_frac.max(axis=1)
    n_cells = marker_tab.sum(axis=1)
else:
    clusters = work.obs[cluster_key].astype(str).value_counts().sort_index().index
    cluster_major_marker = pd.Series("Unknown", index=clusters)
    cluster_major_marker_frac = pd.Series(np.nan, index=clusters)
    n_cells = work.obs[cluster_key].astype(str).value_counts().reindex(clusters).fillna(0).astype(int)

cluster_summary = pd.DataFrame({
    "cluster": cluster_major_marker.index.astype(str),
    "major_marker": cluster_major_marker.values,
    "major_marker_frac": cluster_major_marker_frac.reindex(cluster_major_marker.index).values,
    "n_cells": n_cells.reindex(cluster_major_marker.index).values,
}).set_index("cluster")

# marker-only 结果：不再使用预测标签多数派
cluster_summary["cluster_label_clean"] = cluster_summary["major_marker"].astype(str)
cluster_summary.loc[cluster_summary["cluster_label_clean"].eq("Unknown"), "cluster_label_clean"] = "Uncertain"

cluster_summary = cluster_summary.copy()
cluster_summary["cluster_num"] = cluster_summary.index.astype(int)
cluster_summary = cluster_summary.sort_values("cluster_num")

print(cluster_summary.drop(columns="cluster_num").to_string())
cluster_summary.drop(columns="cluster_num").to_csv(os.path.join(outdir, "mac_infla_cluster_level3_marker_summary.csv"))


                           major_marker  major_marker_frac  n_cells             cluster_label_clean
cluster                                                                                            
0        Chemokine-producing macrophage           0.970332     1146  Chemokine-producing macrophage
1        Chemokine-producing macrophage           0.997802      910  Chemokine-producing macrophage
2        Chemokine-producing macrophage           0.992009      876  Chemokine-producing macrophage
3        Chemokine-producing macrophage           0.682726      851  Chemokine-producing macrophage
4        Chemokine-producing macrophage           0.987639      809  Chemokine-producing macrophage
5        Chemokine-producing macrophage           0.997264      731  Chemokine-producing macrophage
6        Chemokine-producing macrophage           0.995614      684  Chemokine-producing macrophage
7        Chemokine-producing macrophage           0.802360      678  Chemokine-producing macrophage


In [39]:
corrected_annotation = {
    "0": "Pro-inflammatory macrophage",
    "4": "Pro-inflammatory macrophage",
    "8": "Pro-inflammatory macrophage",
    "10": "Pro-inflammatory macrophage",
    "11": "Pro-inflammatory macrophage",
    "13": "Pro-inflammatory macrophage",
    "18": "Pro-inflammatory macrophage",
    "19": "Pro-inflammatory macrophage",
    "20": "Pro-inflammatory macrophage",
    "21": "Pro-inflammatory macrophage",
    "25": "Pro-inflammatory macrophage",
    "27": "Pro-inflammatory macrophage",
    "30": "Pro-inflammatory macrophage"
}
# marker-only 二级注释结果：不再写回/覆盖一级列 cell_type_level1_corrected
level2_col = "cell_type_level3"
cluster_clean_map = cluster_summary["cluster_label_clean"].to_dict()
work.obs[level2_col] = work.obs[cluster_key].astype(str).map(cluster_clean_map).fillna("Unknown")

## 如需人工覆盖 marker-only 结果，再取消下面几行注释。
manual_map = work.obs[cluster_key].astype(str).map(corrected_annotation)
manual_mask = manual_map.notna()
work.obs.loc[manual_mask, level2_col] = manual_map.loc[manual_mask]

print(work.obs[level2_col].value_counts())

cell_type_level3
Chemokine-producing macrophage    12220
Pro-inflammatory macrophage        8036
Name: count, dtype: int64


In [40]:
work.write_h5ad(os.path.join(outdir, "mac_infla_level3_scPoli_recluster_umap.h5ad"))

In [42]:
# 注释后 UMAP：看最终 corrected label
sc.pl.umap(
    work,
    color="cell_type_level3",
    legend_loc="on data",
    frameon=False,
    size=2,
    show=False
)
plt.savefig(os.path.join(outdir, "umap_mac_infla_cell_type_level3.pdf"), bbox_inches="tight")
plt.close()

# # 注释后 UMAP：和 cluster 对照
# sc.pl.umap(
#     work,
#     color=cluster_key,
#     legend_loc="on data",
#     frameon=False,
#     size=2,
#     show=False
# )
# plt.savefig(os.path.join(outdir, "umap_mac_infla_cluster_after_correction.pdf"), bbox_inches="tight")
# plt.close()

# 注释后 UMAP：如果有 dataset，也看 batch/dataset 分布
# if "dataset" in work.obs:
#     sc.pl.umap(
#         work,
#         color="dataset",
#         frameon=False,
#         size=2,
#         show=False
#     )
#     plt.savefig(os.path.join(outdir, "umap_mac_infla_dataset_after_correction.pdf"), bbox_inches="tight")
#     plt.close()

# 注释后 dotplot：按最终 corrected label 分组
sc.pl.dotplot(
    work,
    var_names=marker_dict,
    groupby="cell_type_level2",
    standard_scale="var",
    show=False
)
plt.savefig(os.path.join(outdir, "dot_marker_mac_infla_by_level2_marker.pdf"), bbox_inches="tight")
plt.close()

# # 注释后 dotplot：按 cluster 分组，方便检查每个 cluster 的 marker
# sc.pl.dotplot(
#     work,
#     var_names=marker_dict,
#     groupby=cluster_key,
#     standard_scale="var",
#     show=False
# )
# plt.savefig(os.path.join(outdir, "dot_marker_smc_fib_by_cluster_after_correction.pdf"), bbox_inches="tight")
# plt.close()

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  obs_bool.groupby(level=0).sum() / obs_bo

# mac-foamy

In [11]:
label_col = "cell_type_level2"
cluster_key = "leiden_scpoli_res3"
outdir="./output_mouse/work_0514/mac_foamy/"

In [12]:
def subset_normalize_recluster_scpoli(adata, target_cell_types, outdir, prefix=None, label_col="cell_type_level2", counts_layer="counts", lognorm_layer="lognorm", scpoli_rep="X_scPoli", cluster_key=None, n_neighbors=150, leiden_resolution=3, leiden_random_state=1234, umap_min_dist=1.1, umap_random_state=1234, metric="euclidean", target_sum=1e4, size=2, save_h5ad=True):
    os.makedirs(outdir, exist_ok=True)
    target_cell_types = list(target_cell_types)
    if prefix is None:
        prefix = "_".join([x.lower().replace(" ", "_").replace("/", "_").replace("+", "pos") for x in target_cell_types])
    if cluster_key is None:
        cluster_key = f"leiden_scpoli_res{leiden_resolution}"
    if label_col not in adata.obs:
        raise ValueError(f"{label_col} not found in adata.obs")
    if counts_layer not in adata.layers:
        raise ValueError(f"{counts_layer} not found in adata.layers")
    if scpoli_rep not in adata.obsm:
        raise ValueError(f"{scpoli_rep} not found in adata.obsm")

    print("Target cell types:", target_cell_types)
    print("Original object:", adata)
    work = adata[adata.obs[label_col].astype(str).isin(target_cell_types), :].copy()
    print("Subset object:", work)
    print(f"{label_col}.value_counts() after subsetting:")
    print(work.obs[label_col].value_counts())
    if work.n_obs == 0:
        raise ValueError("No cells selected. Please check target_cell_types and label_col.")

    work.X = work.layers[counts_layer].copy()
    work.layers[counts_layer] = work.X.copy()
    sc.pp.normalize_total(work, target_sum=target_sum)
    sc.pp.log1p(work)
    work.layers[lognorm_layer] = work.X.copy()

    x = work.layers[lognorm_layer]
    v = x.data if sp.issparse(x) else np.ravel(x)
    v = v[:min(len(v), 1000000)]
    print("lognorm min", np.min(v), "max", np.max(v), "integer-like should be False", np.allclose(v, np.round(v)))
    print(f"{label_col}.value_counts() after normalization:")
    print(work.obs[label_col].value_counts())

    print("Formal correction/reclustering:")
    print(f"Only using {label_col}, {scpoli_rep}, and marker-based level2 annotation; not using cell_type_ori / cell_type_level1 as ground truth.")
    print("work object:", work)
    print(f"{label_col}.value_counts() in work:")
    print(work.obs[label_col].value_counts())

    sc.pp.neighbors(work, use_rep=scpoli_rep, n_neighbors=n_neighbors, metric=metric)
    sc.tl.leiden(work, resolution=leiden_resolution, key_added=cluster_key, random_state=leiden_random_state)
    sc.tl.umap(work, min_dist=umap_min_dist, random_state=umap_random_state)

    for c in [cluster_key, label_col]:
        sc.pl.umap(work, color=c, legend_loc="on data", frameon=False, size=size, show=False)
        fig_path = os.path.join(outdir, f"umap_{prefix}_{c}.pdf")
        plt.savefig(fig_path, bbox_inches="tight")
        plt.close()
        print(f"Saved UMAP: {fig_path}")

    if "dataset" in work.obs:
        sc.pl.umap(work, color="dataset", frameon=False, size=size, show=False)
        fig_path = os.path.join(outdir, f"umap_{prefix}_dataset.pdf")
        plt.savefig(fig_path, bbox_inches="tight")
        plt.close()
        print(f"Saved UMAP: {fig_path}")

    print(f"{cluster_key}.value_counts().sort_index():")
    print(work.obs[cluster_key].value_counts().sort_index())

    if save_h5ad:
        save_path = os.path.join(outdir, f"{prefix}_scPoli_recluster_umap.h5ad")
        work.write_h5ad(save_path)
        print(f"Saved to: {save_path}")

    return work

In [13]:
mac_foamy = subset_normalize_recluster_scpoli(
    adata=adata,
    target_cell_types=["Foamy macrophage"],
    outdir=outdir,
    prefix="mac_foamy",
    cluster_key="leiden_scpoli_res3"
)

Target cell types: ['Foamy macrophage']
Original object: AnnData object with n_obs × n_vars = 561576 × 57692
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'tissue', 'cell_type_level1', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'atlas_key', 'cell_type_level1_corrected', 'cell_type_level2'
    var: 'original_gene_names'
    obsm: 'X_scPoli', 'X_umap', 'decontX_GSE131776_10_UMAP', 'decontX_GSE131776_11_UMAP', 'decontX_GSE131776_12_UMAP', 'decontX_GSE131776_13_UMAP', 'decontX_GSE131776_14_UMAP', 'decontX_GSE131776_15_UMAP', 'decontX_GSE131776_16_UMAP', 'decontX_GSE131776_17_UMAP', 'decontX_GSE131776_18_UMAP', 'decontX_GSE131776_1_

Subset object: AnnData object with n_obs × n_vars = 2560 × 57692
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'tissue', 'cell_type_level1', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'atlas_key', 'cell_type_level1_corrected', 'cell_type_level2'
    var: 'original_gene_names'
    obsm: 'X_scPoli', 'X_umap', 'decontX_GSE131776_10_UMAP', 'decontX_GSE131776_11_UMAP', 'decontX_GSE131776_12_UMAP', 'decontX_GSE131776_13_UMAP', 'decontX_GSE131776_14_UMAP', 'decontX_GSE131776_15_UMAP', 'decontX_GSE131776_16_UMAP', 'decontX_GSE131776_17_UMAP', 'decontX_GSE131776_18_UMAP', 'decontX_GSE131776_1_UMAP', 'decontX_GSE131776_2_UMAP', 'decontX_

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/preprocessing/_normalization.py:196: UserWarning: Some cells have zero counts
  warn(UserWarning('Some cells have zero counts'))


    finished (0:00:00)
lognorm min 0.17235006 max 7.63398 integer-like should be False False
cell_type_level2.value_counts() after normalization:
cell_type_level2
Foamy macrophage    2560
Name: count, dtype: int64
Formal correction/reclustering:
Only using cell_type_level2, X_scPoli, and marker-based level2 annotation; not using cell_type_ori / cell_type_level1 as ground truth.
work object: AnnData object with n_obs × n_vars = 2560 × 57692
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'tissue', 'cell_type_level1', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'atlas_key', 'cell_type_level1_corrected', 'cell_type_level2'
    var: '

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_mouse/work_0514/mac_foamy/umap_mac_foamy_leiden_scpoli_res3.pdf


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_mouse/work_0514/mac_foamy/umap_mac_foamy_cell_type_level2.pdf


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_mouse/work_0514/mac_foamy/umap_mac_foamy_dataset.pdf
leiden_scpoli_res3.value_counts().sort_index():
leiden_scpoli_res3
0     155
1     134
2     118
3     118
4     118
5     116
6     114
7     111
8     106
9     105
10    103
11     95
12     89
13     87
14     87
15     82
16     81
17     81
18     76
19     68
20     66
21     66
22     60
23     57
24     55
25     53
26     51
27     44
28     35
29     29
Name: count, dtype: int64
Saved to: ./output_mouse/work_0514/mac_foamy/mac_foamy_scPoli_recluster_umap.h5ad


In [ ]:
label_col = "cell_type_level2"
cluster_key = "leiden_scpoli_res3"
outdir="./output_mouse/work_0514/mac_foamy/"
work = sc.read_h5ad(os.path.join(outdir, "mac_foamy_scPoli_recluster_umap.h5ad"))
work

AnnData object with n_obs × n_vars = 12463 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected', 'cell_type_level2', 'leiden_scpoli_res3'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'cell_type_level2_colors', 'dataset_colors', 'leiden', 'leiden_scpoli_res3_colors', 'log1p', 'neighbors', 'umap'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'lognorm', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
    obsp: 'connectivities', 'distances'

In [44]:
# 如果 h5ad 里 var_names 不是 gene symbol，先改成 gene symbol
work.var_names = work.var["original_gene_names"].astype(str).values
work.var_names_make_unique()

# marker-based sanity check：用 marker 给每个细胞打 marker_top 标签
marker_dict = {
    "Cholesteryl-Ester-rich Foamy macrophage": ['PLIN2', 'PLIN3', 'CD36', 'SOAT1','ACAT1', 'FABP5', 'APOC2', 'LPL', 'NCEH1', 'NPC2'],
    "Triglyceride/Vesicle-Trafficking-biased Foamy macrophage": ['FABP4', 'PLTP', 'GPNMB', 'CD63', 'DGAT1', 'RAB7B'],
}

marker_dict = {k: [g for g in v if g in work.var_names] for k, v in marker_dict.items()}
marker_dict = {k: v for k, v in marker_dict.items() if len(v) > 0}

print("marker genes used:")
for k, v in marker_dict.items():
    print(k, v)

def mean_expr_by_genes(adata, genes, layer="lognorm"):
    X = adata[:, genes].layers[layer] if layer in adata.layers else adata[:, genes].X
    return np.asarray(X.mean(axis=1)).ravel()

score_cols = []
score_col_to_label = {}

for ct, genes in marker_dict.items():
    safe = ct.replace(" ", "_").replace("/", "_")
    col = f"marker_score__{safe}"
    work.obs[col] = mean_expr_by_genes(work, genes, layer="lognorm")
    score_cols.append(col)
    score_col_to_label[col] = ct

if len(score_cols) > 0:
    score_mat = work.obs[score_cols].to_numpy(dtype=float)
    order = np.argsort(score_mat, axis=1)
    top_idx = order[:, -1]
    second_idx = order[:, -2] if len(score_cols) > 1 else order[:, -1]

    score_labels = np.array([score_col_to_label[c] for c in score_cols])
    work.obs["marker_top"] = score_labels[top_idx]
    work.obs["marker_top_score"] = score_mat[np.arange(work.n_obs), top_idx]
    work.obs["marker_second_score"] = score_mat[np.arange(work.n_obs), second_idx]
    work.obs["marker_margin"] = work.obs["marker_top_score"] - work.obs["marker_second_score"]
    work.obs["marker_disagree"] = ((work.obs["marker_top"].astype(str) != work.obs[label_col].astype(str)) & (work.obs["marker_margin"].astype(float) > 0.1))

    sc.pl.dotplot(work, var_names=marker_dict, groupby=label_col, standard_scale="var", show=False)
    plt.savefig(os.path.join(outdir, "dot_marker_mac_foamy_by_level2_marker.pdf"), bbox_inches="tight")
    plt.close()

    sc.pl.dotplot(work, var_names=marker_dict, groupby=cluster_key, standard_scale="var", show=False)
    plt.savefig(os.path.join(outdir, "dot_marker_mac_foamy_by_cluster.pdf"), bbox_inches="tight")
    plt.close()
else:
    work.obs["marker_top"] = "Unknown"
    work.obs["marker_margin"] = np.nan
    work.obs["marker_disagree"] = False
    print("no marker genes found; skip marker sanity check")

marker genes used:
Cholesteryl-Ester-rich Foamy macrophage ['PLIN2', 'PLIN3', 'CD36', 'SOAT1', 'ACAT1', 'FABP5', 'APOC2', 'LPL', 'NCEH1', 'NPC2']
Triglyceride/Vesicle-Trafficking-biased Foamy macrophage ['FABP4', 'PLTP', 'GPNMB', 'CD63', 'DGAT1', 'RAB7B']


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  obs_bool.groupby(level=0).sum() / obs_bool.groupby(level=0).count()
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:178: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dot_color_df = self.obs_tidy.groupby(level=0).mean()
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:747: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap', 'norm' will be ignored
  dot

In [45]:
# cluster-level 二级注释：不用预测标签多数派，只看每个 scPoli cluster 内的 marker_top 多数派
if "marker_top" in work.obs and (work.obs["marker_top"].astype(str) != "Unknown").any():
    marker_tab = pd.crosstab(work.obs[cluster_key].astype(str), work.obs["marker_top"].astype(str))
    marker_frac = marker_tab.div(marker_tab.sum(axis=1), axis=0)
    cluster_major_marker = marker_tab.idxmax(axis=1)
    cluster_major_marker_frac = marker_frac.max(axis=1)
    n_cells = marker_tab.sum(axis=1)
else:
    clusters = work.obs[cluster_key].astype(str).value_counts().sort_index().index
    cluster_major_marker = pd.Series("Unknown", index=clusters)
    cluster_major_marker_frac = pd.Series(np.nan, index=clusters)
    n_cells = work.obs[cluster_key].astype(str).value_counts().reindex(clusters).fillna(0).astype(int)

cluster_summary = pd.DataFrame({
    "cluster": cluster_major_marker.index.astype(str),
    "major_marker": cluster_major_marker.values,
    "major_marker_frac": cluster_major_marker_frac.reindex(cluster_major_marker.index).values,
    "n_cells": n_cells.reindex(cluster_major_marker.index).values,
}).set_index("cluster")

# marker-only 结果：不再使用预测标签多数派
cluster_summary["cluster_label_clean"] = cluster_summary["major_marker"].astype(str)
cluster_summary.loc[cluster_summary["cluster_label_clean"].eq("Unknown"), "cluster_label_clean"] = "Uncertain"

cluster_summary = cluster_summary.copy()
cluster_summary["cluster_num"] = cluster_summary.index.astype(int)
cluster_summary = cluster_summary.sort_values("cluster_num")

print(cluster_summary.drop(columns="cluster_num").to_string())
cluster_summary.drop(columns="cluster_num").to_csv(os.path.join(outdir, "mac_foamy_cluster_level3_marker_summary.csv"))


                                                     major_marker  major_marker_frac  n_cells                                       cluster_label_clean
cluster                                                                                                                                                
0        Triglyceride/Vesicle-Trafficking-biased Foamy macrophage           0.655620      694  Triglyceride/Vesicle-Trafficking-biased Foamy macrophage
1                         Cholesteryl-Ester-rich Foamy macrophage           0.539359      686                   Cholesteryl-Ester-rich Foamy macrophage
2        Triglyceride/Vesicle-Trafficking-biased Foamy macrophage           0.746544      651  Triglyceride/Vesicle-Trafficking-biased Foamy macrophage
3                         Cholesteryl-Ester-rich Foamy macrophage           0.676704      631                   Cholesteryl-Ester-rich Foamy macrophage
4                         Cholesteryl-Ester-rich Foamy macrophage           0.557508    

In [46]:
corrected_annotation = {
    "1" : "Cholesteryl-Ester-rich Foamy macrophage",
    "6" : "Triglyceride/Vesicle-Trafficking-biased Foamy macrophage",
    "12" : "Triglyceride/Vesicle-Trafficking-biased Foamy macrophage",
    "20" : "Cholesteryl-Ester-rich Foamy macrophage",
    "21" : "Triglyceride/Vesicle-Trafficking-biased Foamy macrophage",
    "23" : "Triglyceride/Vesicle-Trafficking-biased Foamy macrophage",
    "26" : "Triglyceride/Vesicle-Trafficking-biased Foamy macrophage"
}
# marker-only 二级注释结果：不再写回/覆盖一级列 cell_type_level1_corrected
level2_col = "cell_type_level3"
cluster_clean_map = cluster_summary["cluster_label_clean"].to_dict()
work.obs[level2_col] = work.obs[cluster_key].astype(str).map(cluster_clean_map).fillna("Unknown")

# 如需人工覆盖 marker-only 结果，再取消下面几行注释。
manual_map = work.obs[cluster_key].astype(str).map(corrected_annotation)
manual_mask = manual_map.notna()
work.obs.loc[manual_mask, level2_col] = manual_map.loc[manual_mask]

print(work.obs[level2_col].value_counts())

cell_type_level3
Triglyceride/Vesicle-Trafficking-biased Foamy macrophage    8934
Cholesteryl-Ester-rich Foamy macrophage                     3529
Name: count, dtype: int64


In [47]:
work.write_h5ad(os.path.join(outdir, "mac_foamy_level3_scPoli_recluster_umap.h5ad"))

In [48]:
# 注释后 UMAP：看最终 corrected label
sc.pl.umap(
    work,
    color="cell_type_level3",
    legend_loc="on data",
    frameon=False,
    size=2,
    show=False
)
plt.savefig(os.path.join(outdir, "umap_mac_foamy_cell_type_level2_marker.pdf"), bbox_inches="tight")
plt.close()

# # 注释后 UMAP：和 cluster 对照
# sc.pl.umap(
#     work,
#     color=cluster_key,
#     legend_loc="on data",
#     frameon=False,
#     size=2,
#     show=False
# )
# plt.savefig(os.path.join(outdir, "umap_tc_cluster_after_correction.pdf"), bbox_inches="tight")
# plt.close()

# 注释后 UMAP：如果有 dataset，也看 batch/dataset 分布
# if "dataset" in work.obs:
#     sc.pl.umap(
#         work,
#         color="dataset",
#         frameon=False,
#         size=2,
#         show=False
#     )
#     plt.savefig(os.path.join(outdir, "umap_tc_dataset_after_correction.pdf"), bbox_inches="tight")
#     plt.close()

# 注释后 dotplot：按最终 corrected label 分组
sc.pl.dotplot(
    work,
    var_names=marker_dict,
    groupby="cell_type_level3",
    standard_scale="var",
    show=False
)
plt.savefig(os.path.join(outdir, "dot_marker_mac_foamy_by_level3.pdf"), bbox_inches="tight")
plt.close()

# # 注释后 dotplot：按 cluster 分组，方便检查每个 cluster 的 marker
# sc.pl.dotplot(
#     work,
#     var_names=marker_dict,
#     groupby=cluster_key,
#     standard_scale="var",
#     show=False
# )
# plt.savefig(os.path.join(outdir, "dot_marker_tc_by_cluster_after_correction.pdf"), bbox_inches="tight")
# plt.close()

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  obs_bool.groupby(level=0).sum() / obs_bo

# mac-homeo

In [14]:
label_col = "cell_type_level2"
cluster_key = "leiden_scpoli_res3"
outdir="./output_mouse/work_0514/mac_homeostatic"

In [15]:
def subset_normalize_recluster_scpoli(adata, target_cell_types, outdir, prefix=None, label_col="cell_type_level2", counts_layer="counts", lognorm_layer="lognorm", scpoli_rep="X_scPoli", cluster_key=None, n_neighbors=150, leiden_resolution=3, leiden_random_state=1234, umap_min_dist=1.1, umap_random_state=1234, metric="euclidean", target_sum=1e4, size=2, save_h5ad=True):
    os.makedirs(outdir, exist_ok=True)
    target_cell_types = list(target_cell_types)
    if prefix is None:
        prefix = "_".join([x.lower().replace(" ", "_").replace("/", "_").replace("+", "pos") for x in target_cell_types])
    if cluster_key is None:
        cluster_key = f"leiden_scpoli_res{leiden_resolution}"
    if label_col not in adata.obs:
        raise ValueError(f"{label_col} not found in adata.obs")
    if counts_layer not in adata.layers:
        raise ValueError(f"{counts_layer} not found in adata.layers")
    if scpoli_rep not in adata.obsm:
        raise ValueError(f"{scpoli_rep} not found in adata.obsm")

    print("Target cell types:", target_cell_types)
    print("Original object:", adata)
    work = adata[adata.obs[label_col].astype(str).isin(target_cell_types), :].copy()
    print("Subset object:", work)
    print(f"{label_col}.value_counts() after subsetting:")
    print(work.obs[label_col].value_counts())
    if work.n_obs == 0:
        raise ValueError("No cells selected. Please check target_cell_types and label_col.")

    work.X = work.layers[counts_layer].copy()
    work.layers[counts_layer] = work.X.copy()
    sc.pp.normalize_total(work, target_sum=target_sum)
    sc.pp.log1p(work)
    work.layers[lognorm_layer] = work.X.copy()

    x = work.layers[lognorm_layer]
    v = x.data if sp.issparse(x) else np.ravel(x)
    v = v[:min(len(v), 1000000)]
    print("lognorm min", np.min(v), "max", np.max(v), "integer-like should be False", np.allclose(v, np.round(v)))
    print(f"{label_col}.value_counts() after normalization:")
    print(work.obs[label_col].value_counts())

    print("Formal correction/reclustering:")
    print(f"Only using {label_col}, {scpoli_rep}, and marker-based level2 annotation; not using cell_type_ori / cell_type_level1 as ground truth.")
    print("work object:", work)
    print(f"{label_col}.value_counts() in work:")
    print(work.obs[label_col].value_counts())

    sc.pp.neighbors(work, use_rep=scpoli_rep, n_neighbors=n_neighbors, metric=metric)
    sc.tl.leiden(work, resolution=leiden_resolution, key_added=cluster_key, random_state=leiden_random_state)
    sc.tl.umap(work, min_dist=umap_min_dist, random_state=umap_random_state)

    for c in [cluster_key, label_col]:
        sc.pl.umap(work, color=c, legend_loc="on data", frameon=False, size=size, show=False)
        fig_path = os.path.join(outdir, f"umap_{prefix}_{c}.pdf")
        plt.savefig(fig_path, bbox_inches="tight")
        plt.close()
        print(f"Saved UMAP: {fig_path}")

    if "dataset" in work.obs:
        sc.pl.umap(work, color="dataset", frameon=False, size=size, show=False)
        fig_path = os.path.join(outdir, f"umap_{prefix}_dataset.pdf")
        plt.savefig(fig_path, bbox_inches="tight")
        plt.close()
        print(f"Saved UMAP: {fig_path}")

    print(f"{cluster_key}.value_counts().sort_index():")
    print(work.obs[cluster_key].value_counts().sort_index())

    if save_h5ad:
        save_path = os.path.join(outdir, f"{prefix}_scPoli_recluster_umap.h5ad")
        work.write_h5ad(save_path)
        print(f"Saved to: {save_path}")

    return work

In [ ]:
mac_homeostatic = subset_normalize_recluster_scpoli(
    adata=adata,
    target_cell_types=["Homeostatic/Resident macrophage"],
    outdir=outdir,
    prefix="mac_homeostatic",
    cluster_key="leiden_scpoli_res3"
)

Target cell types: ['Homeostatic/Resident macrophage']
Original object: AnnData object with n_obs × n_vars = 561576 × 57692
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'tissue', 'cell_type_level1', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'atlas_key', 'cell_type_level1_corrected', 'cell_type_level2'
    var: 'original_gene_names'
    obsm: 'X_scPoli', 'X_umap', 'decontX_GSE131776_10_UMAP', 'decontX_GSE131776_11_UMAP', 'decontX_GSE131776_12_UMAP', 'decontX_GSE131776_13_UMAP', 'decontX_GSE131776_14_UMAP', 'decontX_GSE131776_15_UMAP', 'decontX_GSE131776_16_UMAP', 'decontX_GSE131776_17_UMAP', 'decontX_GSE131776_18_UMAP', 'decon

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/preprocessing/_normalization.py:196: UserWarning: Some cells have zero counts
  warn(UserWarning('Some cells have zero counts'))


    finished (0:00:10)
lognorm min 0.24109001 max 9.210441 integer-like should be False False
cell_type_level2.value_counts() after normalization:
cell_type_level2
Homeostatic/Resident macrophage    71132
Name: count, dtype: int64
Formal correction/reclustering:
Only using cell_type_level2, X_scPoli, and marker-based level2 annotation; not using cell_type_ori / cell_type_level1 as ground truth.
work object: AnnData object with n_obs × n_vars = 71132 × 57692
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'tissue', 'cell_type_level1', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'atlas_key', 'cell_type_level1_corrected', 'cell_type_

In [ ]:
label_col = "cell_type_level2"
cluster_key = "leiden_scpoli_res3"
outdir="./output_mouse/work_0514/mac_homeostatic"
work = sc.read_h5ad(os.path.join(outdir, "mac_homeostatic_scPoli_recluster_umap.h5ad"))
work

AnnData object with n_obs × n_vars = 69184 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected', 'cell_type_level2', 'leiden_scpoli_res3'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'cell_type_level2_colors', 'dataset_colors', 'leiden', 'leiden_scpoli_res3_colors', 'log1p', 'neighbors', 'umap'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'lognorm', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
    obsp: 'connectivities', 'distances'

In [52]:
# 如果 h5ad 里 var_names 不是 gene symbol，先改成 gene symbol
work.var_names = work.var["original_gene_names"].astype(str).values
work.var_names_make_unique()

# marker-based sanity check：用 marker 给每个细胞打 marker_top 标签
marker_dict = {
    "Resident-like macrophage": ['LYVE1','CD206', 'F13A1', 'SEPP1', 'IGF1', 'GAS6', 'MERTK', 'STAB1','C1QA','C1QB','C1QC'],
    "Scavenger anti-inflammatory macrophage": ['CD163', 'MRC1', 'FCRLS', 'CBR2', 'MS4A4A', 'MARCO'],
}

marker_dict = {k: [g for g in v if g in work.var_names] for k, v in marker_dict.items()}
marker_dict = {k: v for k, v in marker_dict.items() if len(v) > 0}

print("marker genes used:")
for k, v in marker_dict.items():
    print(k, v)

def mean_expr_by_genes(adata, genes, layer="lognorm"):
    X = adata[:, genes].layers[layer] if layer in adata.layers else adata[:, genes].X
    return np.asarray(X.mean(axis=1)).ravel()

score_cols = []
score_col_to_label = {}

for ct, genes in marker_dict.items():
    safe = ct.replace(" ", "_").replace("/", "_")
    col = f"marker_score__{safe}"
    work.obs[col] = mean_expr_by_genes(work, genes, layer="lognorm")
    score_cols.append(col)
    score_col_to_label[col] = ct

if len(score_cols) > 0:
    score_mat = work.obs[score_cols].to_numpy(dtype=float)
    order = np.argsort(score_mat, axis=1)
    top_idx = order[:, -1]
    second_idx = order[:, -2] if len(score_cols) > 1 else order[:, -1]

    score_labels = np.array([score_col_to_label[c] for c in score_cols])
    work.obs["marker_top"] = score_labels[top_idx]
    work.obs["marker_top_score"] = score_mat[np.arange(work.n_obs), top_idx]
    work.obs["marker_second_score"] = score_mat[np.arange(work.n_obs), second_idx]
    work.obs["marker_margin"] = work.obs["marker_top_score"] - work.obs["marker_second_score"]
    work.obs["marker_disagree"] = ((work.obs["marker_top"].astype(str) != work.obs[label_col].astype(str)) & (work.obs["marker_margin"].astype(float) > 0.1))

    sc.pl.dotplot(work, var_names=marker_dict, groupby=label_col, standard_scale="var", show=False)
    plt.savefig(os.path.join(outdir, "dot_marker_mac_homeostatic_by_level2.pdf"), bbox_inches="tight")
    plt.close()

    sc.pl.dotplot(work, var_names=marker_dict, groupby=cluster_key, standard_scale="var", show=False)
    plt.savefig(os.path.join(outdir, "dot_marker_mac_homeostatic_by_cluster.pdf"), bbox_inches="tight")
    plt.close()
else:
    work.obs["marker_top"] = "Unknown"
    work.obs["marker_margin"] = np.nan
    work.obs["marker_disagree"] = False
    print("no marker genes found; skip marker sanity check")

marker genes used:
Resident-like macrophage ['LYVE1', 'F13A1', 'IGF1', 'GAS6', 'MERTK', 'STAB1', 'C1QA', 'C1QB', 'C1QC']
Scavenger anti-inflammatory macrophage ['CD163', 'MRC1', 'MS4A4A', 'MARCO']


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  obs_bool.groupby(level=0).sum() / obs_bool.groupby(level=0).count()
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:178: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dot_color_df = self.obs_tidy.groupby(level=0).mean()
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:747: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap', 'norm' will be ignored
  dot

In [53]:
# cluster-level 二级注释：不用预测标签多数派，只看每个 scPoli cluster 内的 marker_top 多数派
if "marker_top" in work.obs and (work.obs["marker_top"].astype(str) != "Unknown").any():
    marker_tab = pd.crosstab(work.obs[cluster_key].astype(str), work.obs["marker_top"].astype(str))
    marker_frac = marker_tab.div(marker_tab.sum(axis=1), axis=0)
    cluster_major_marker = marker_tab.idxmax(axis=1)
    cluster_major_marker_frac = marker_frac.max(axis=1)
    n_cells = marker_tab.sum(axis=1)
else:
    clusters = work.obs[cluster_key].astype(str).value_counts().sort_index().index
    cluster_major_marker = pd.Series("Unknown", index=clusters)
    cluster_major_marker_frac = pd.Series(np.nan, index=clusters)
    n_cells = work.obs[cluster_key].astype(str).value_counts().reindex(clusters).fillna(0).astype(int)

cluster_summary = pd.DataFrame({
    "cluster": cluster_major_marker.index.astype(str),
    "major_marker": cluster_major_marker.values,
    "major_marker_frac": cluster_major_marker_frac.reindex(cluster_major_marker.index).values,
    "n_cells": n_cells.reindex(cluster_major_marker.index).values,
}).set_index("cluster")

# marker-only 结果：不再使用预测标签多数派
cluster_summary["cluster_label_clean"] = cluster_summary["major_marker"].astype(str)
cluster_summary.loc[cluster_summary["cluster_label_clean"].eq("Unknown"), "cluster_label_clean"] = "Uncertain"

cluster_summary = cluster_summary.copy()
cluster_summary["cluster_num"] = cluster_summary.index.astype(int)
cluster_summary = cluster_summary.sort_values("cluster_num")

print(cluster_summary.drop(columns="cluster_num").to_string())
cluster_summary.drop(columns="cluster_num").to_csv(os.path.join(outdir, "mac_homeostatic_cluster_level2_marker_summary.csv"))


                     major_marker  major_marker_frac  n_cells       cluster_label_clean
cluster                                                                                
0        Resident-like macrophage           0.573659     3075  Resident-like macrophage
1        Resident-like macrophage           0.866854     2486  Resident-like macrophage
2        Resident-like macrophage           0.891057     2460  Resident-like macrophage
3        Resident-like macrophage           0.896393     2384  Resident-like macrophage
4        Resident-like macrophage           0.924203     2322  Resident-like macrophage
5        Resident-like macrophage           0.955263     2280  Resident-like macrophage
6        Resident-like macrophage           0.784147     2233  Resident-like macrophage
7        Resident-like macrophage           0.917493     2218  Resident-like macrophage
8        Resident-like macrophage           0.801454     2201  Resident-like macrophage
9        Resident-like macrophag

In [55]:
corrected_annotation = {
    "0": "Scavenger anti-inflammatory macrophage",
    "6": "Scavenger anti-inflammatory macrophage",
    "8": "Scavenger anti-inflammatory macrophage",
    "9": "Scavenger anti-inflammatory macrophage",
    "10": "Scavenger anti-inflammatory macrophage",
    "12": "Scavenger anti-inflammatory macrophage",
    "13": "Scavenger anti-inflammatory macrophage",
    "15": "Scavenger anti-inflammatory macrophage",
    "18": "Scavenger anti-inflammatory macrophage",
    "19": "Scavenger anti-inflammatory macrophage",
    "20": "Scavenger anti-inflammatory macrophage",
    "21": "Scavenger anti-inflammatory macrophage",
    "26": "Scavenger anti-inflammatory macrophage",
    "27": "Scavenger anti-inflammatory macrophage",
    "30": "Scavenger anti-inflammatory macrophage",
    "31": "Scavenger anti-inflammatory macrophage",
    "39": "Scavenger anti-inflammatory macrophage"
   
}
# marker-only 二级注释结果：不再写回/覆盖一级列 cell_type_level1_corrected
level2_col = "cell_type_level3"
cluster_clean_map = cluster_summary["cluster_label_clean"].to_dict()
work.obs[level2_col] = work.obs[cluster_key].astype(str).map(cluster_clean_map).fillna("Unknown")

# 如需人工覆盖 marker-only 结果，再取消下面几行注释。
manual_map = work.obs[cluster_key].astype(str).map(corrected_annotation)
manual_mask = manual_map.notna()
work.obs.loc[manual_mask, level2_col] = manual_map.loc[manual_mask]

print(work.obs[level2_col].value_counts())

cell_type_level3
Resident-like macrophage                  38792
Scavenger anti-inflammatory macrophage    30392
Name: count, dtype: int64


In [56]:
work.write_h5ad(os.path.join(outdir, "mac_homeostatic_level3_scPoli_recluster_umap.h5ad"))

In [57]:
# 注释后 UMAP：看最终 corrected label
sc.pl.umap(
    work,
    color="cell_type_level3",
    legend_loc="on data",
    frameon=False,
    size=2,
    show=False
)
plt.savefig(os.path.join(outdir, "umap_mac_homeostatic_cell_type_level3.pdf"), bbox_inches="tight")
plt.close()

# # 注释后 UMAP：和 cluster 对照
# sc.pl.umap(
#     work,
#     color=cluster_key,
#     legend_loc="on data",
#     frameon=False,
#     size=2,
#     show=False
# )
# plt.savefig(os.path.join(outdir, "umap_ec_cluster_after_correction.pdf"), bbox_inches="tight")
# plt.close()

# 注释后 UMAP：如果有 dataset，也看 batch/dataset 分布
# if "dataset" in work.obs:
#     sc.pl.umap(
#         work,
#         color="dataset",
#         frameon=False,
#         size=2,
#         show=False
#     )
#     plt.savefig(os.path.join(outdir, "umap_ec_dataset_after_correction.pdf"), bbox_inches="tight")
#     plt.close()

# 注释后 dotplot：按最终 corrected label 分组
sc.pl.dotplot(
    work,
    var_names=marker_dict,
    groupby="cell_type_level3",
    standard_scale="var",
    show=False
)
plt.savefig(os.path.join(outdir, "dot_marker_mac_homeostatic_by_level3.pdf"), bbox_inches="tight")
plt.close()

# # 注释后 dotplot：按 cluster 分组，方便检查每个 cluster 的 marker
# sc.pl.dotplot(
#     work,
#     var_names=marker_dict,
#     groupby=cluster_key,
#     standard_scale="var",
#     show=False
# )
# plt.savefig(os.path.join(outdir, "dot_marker_ec_by_cluster_after_correction.pdf"), bbox_inches="tight")
# plt.close()

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  obs_bool.groupby(level=0).sum() / obs_bo

# 合并

In [2]:
outdir="/home/lixiangyu/zr/Annotate/ANNOTATE_new/8_annotate_level2/0514_rename_level2/output_allhuman/work_0514"
adata = sc.read_h5ad(os.path.join(outdir,"scPoli_concat_level2_marker_allhuman.h5ad"))
adata

AnnData object with n_obs × n_vars = 1015699 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected', 'cell_type_level2'
    var: 'original_gene_names', 'ensembl_id'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'

In [3]:
adata.obs["cell_type_level2"].value_counts()

cell_type_level2
CD4 T cell                               228073
CD8 T cell                               127439
Fibroblast                                87730
Neutrophil                                83219
B cell                                    78088
Homeostatic/Resident macrophage           69184
Smooth muscle cell                        57935
Natural killer cell                       53467
Lymphatic endothelial cell                43449
Classical monocyte                        29038
Arterial endothelial cell                 21668
cDC1                                      20734
Inflammatory macrophage                   20256
Pericyte                                  18985
Fibromyocyte                              14546
Foamy macrophage                          12463
Plasma cell                               11518
cDC2                                       9944
Mast cell                                  6959
Erythrocyte/Erythroid                      6884
Intermediate monocyte  

In [ ]:
outdir="/home/lixiangyu/zr/Annotate/ANNOTATE_new/9_annotate_level3/0514_rename_level3/output_mouse/work_0514"

adata_ec_arterial = sc.read_h5ad(os.path.join(outdir, "ec_arterial/ec_arterial_level3_scPoli_recluster_umap.h5ad"))
adata_mac_infla = sc.read_h5ad(os.path.join(outdir, "mac_infla/mac_infla_level3_scPoli_recluster_umap.h5ad"))
adata_mac_foamy = sc.read_h5ad(os.path.join(outdir, "mac_foamy/mac_foamy_level3_scPoli_recluster_umap.h5ad"))
adata_mac_homeostatic = sc.read_h5ad(os.path.join(outdir, "mac_homeostatic/mac_homeostatic_level3_scPoli_recluster_umap.h5ad"))

In [5]:
adata_list = [adata_ec_arterial, adata_mac_infla, adata_mac_foamy, adata_mac_homeostatic]

In [6]:
adata_concat = anndata.concat(adata_list, join='outer', fill_value=0.0)

In [7]:
adata_concat.write(os.path.join(outdir,"scPoli_concat_level3.h5ad"))
adata_concat

AnnData object with n_obs × n_vars = 123571 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected', 'cell_type_level2', 'leiden_scpoli_res3', 'marker_score__Quiescent_state_endothelial', 'marker_score__Pro-angiogenic_endothelial,_tip_state', 'marker_score__Pro-angiogenic_endothelial,_stalk_state', 'marker_score__EndoMT', 'marker_score__Inflammatory_endothelial', 'marker_top', 'marker_top_score', 'marker_second_score', 'marker_margin', 'marker_disagree', 'cell_type_level3', 'marker_score__Pro

In [ ]:
# adata_concat = sc.read_h5ad(os.path.join(outdir,"scPoli_concat_level3.h5ad"))
# adata_concat

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


AnnData object with n_obs × n_vars = 347121 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'leiden_scpoli_res3', 'marker_score__B_cell', 'marker_score__T_cell', 'marker_score__Natural_killer_cell', 'marker_score__Dendritic_cell', 'marker_score__Macrophage', 'marker_score__Monocyte', 'marker_score__Mast_cell', 'marker_score__Erythrocyte_Erythroid', 'marker_score__Neutrophil', 'marker_score__Basophil', 'marker_score__Endothelial_cell', 'marker_score__Fibroblast', 'marker_score__Smooth_muscle_cell', 'marker_score__

In [8]:
adata_concat.obs_names

Index(['1_JD__AAAGTCCAGTTTGGCT-1', '1_JD__AAATGGACAGTAGGAC-1',
       '1_JD__AAGCATCTCGTTGTTT-1', '1_JD__AAGTCGTAGCATGAAT-1',
       '1_JD__AATCGTGGTCGGTGAA-1', '1_JD__AATGACCAGGATTCCT-1',
       '1_JD__ACAGAAATCCGCACGA-1', '1_JD__ACAGCCGGTCGTGATT-1',
       '1_JD__AGCCAGCTCGTAGGGA-1', '1_JD__AGCGTCGCATATGAAG-1',
       ...
       'GSE196943_8__ATGGGAGTCAATCACG_H_plaque',
       'GSE196943_8__ATTGGTGGTCAGAAGC_H_plaque',
       'GSE196943_8__CAGATCAAGTCATGCT_H_plaque',
       'GSE196943_8__CAGTCCTGTACACCGC_H_plaque',
       'GSE196943_8__CCCTCCTGTTTCCACC_H_plaque',
       'GSE196943_8__CGAGCCAAGATGCCAG_H_plaque',
       'GSE196943_8__CGGACACGTCAACATC_H_plaque',
       'GSE196943_8__GCTTCCAAGCGACGTA_H_plaque',
       'GSE196943_8__GTCAAGTTCTCGCTTG_H_plaque',
       'GSE196943_8__CAGATCAGTAAACACA_H_plaque'],
      dtype='object', name='match_id', length=123571)

In [9]:
# Extract barcodes and cell_type_level3 values
barcodes = adata_concat.obs_names
cell_types_level3 = adata_concat.obs["cell_type_level3"]

# Create the mapping
mapping = dict(zip(barcodes, cell_types_level3))

In [10]:
adata.obs_names

Index(['1_JD__AAACCCAAGAGGTTAT-1', '1_JD__AAACCCAAGCAACAAT-1',
       '1_JD__AAACCCAAGGAGTCTG-1', '1_JD__AAACCCACAACTCATG-1',
       '1_JD__AAACCCACAGCTCTGG-1', '1_JD__AAACCCAGTCAAGGCA-1',
       '1_JD__AAACCCAGTCAATCTG-1', '1_JD__AAACCCAGTCACCCTT-1',
       '1_JD__AAACCCAGTGCATTAC-1', '1_JD__AAACCCAGTGCGGCTT-1',
       ...
       'GSE196943_8__ATAACGCAGTACGCCC_H_plaque',
       'GSE196943_8__CACATAGGTGTCTGAT_H_plaque',
       'GSE196943_8__CCATTCGAGTAGCCGA_H_plaque',
       'GSE196943_8__GAATAAGAGAAAGTGG_H_plaque',
       'GSE196943_8__GACCAATCATGGTAGG_H_plaque',
       'GSE196943_8__GACCTGGGTCTTCAAG_H_plaque',
       'GSE196943_8__TGGGCGTTCTTCTGGC_H_plaque',
       'GSE196943_8__GAGCAGATCAGAGGTG_H_plaque',
       'GSE196943_8__GAGTCCGGTTCTGGTA_H_plaque',
       'GSE196943_8__TCATTTGAGTTGTCGT_H_plaque'],
      dtype='object', name='match_id', length=1015699)

In [11]:
# init new column
adata.obs["cell_type_level3"] = "no map"

In [12]:
### more fast
if "cell_type_level3" not in adata.obs.columns:
    adata.obs["cell_type_level3"] = pd.NA

mapped = adata.obs.index.to_series().map(mapping)
mask = mapped.notna()
adata.obs.loc[mask, "cell_type_level3"] = mapped[mask].to_numpy()

In [13]:
adata.obs["cell_type_level3"].value_counts()

cell_type_level3
no map                                                      892128
Resident-like macrophage                                     38792
Scavenger anti-inflammatory macrophage                       30392
Quiescent state endothelial                                  18492
Chemokine-producing macrophage                               12220
Triglyceride/Vesicle-Trafficking-biased Foamy macrophage      8934
Pro-inflammatory macrophage                                   8036
Cholesteryl-Ester-rich Foamy macrophage                       3529
Inflammatory endothelial                                      3079
EndoMT                                                          97
Name: count, dtype: int64

In [15]:
# adata = adata[adata.obs['cell_type_level3'] != 'Undefine'].copy()
# adata

In [16]:
mask = adata.obs["cell_type_level3"] == "no map"
adata.obs.loc[mask, "cell_type_level3"] = adata.obs.loc[mask, "cell_type_level2"]

In [17]:
adata.obs["cell_type_level3"].value_counts()

cell_type_level3
CD4 T cell                                                  228073
CD8 T cell                                                  127439
Fibroblast                                                   87730
Neutrophil                                                   83219
B cell                                                       78088
Smooth muscle cell                                           57935
Natural killer cell                                          53467
Lymphatic endothelial cell                                   43449
Resident-like macrophage                                     38792
Scavenger anti-inflammatory macrophage                       30392
Classical monocyte                                           29038
cDC1                                                         20734
Pericyte                                                     18985
Quiescent state endothelial                                  18492
Fibromyocyte                                 

In [ ]:
adata.write(os.path.join(outdir,"scPoli_concat_level3_marker_allmouse.h5ad"))
adata

AnnData object with n_obs × n_vars = 1015699 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected', 'cell_type_level2', 'cell_type_level3'
    var: 'original_gene_names', 'ensembl_id'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'